# 00 — Setup

**Run once per machine** (or after clearing the Brightway project).

This notebook:
1. Restores the `tz_cotton` Brightway project from an Activity Browser backup
2. Loads all spatial datasets and registers geocollections
3. Registers all LCIA methods (land use, water, N/P eutrophication, climate change)

In [1]:
import sys, os
# Make src/ importable from the notebook
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

_prefix = sys.prefix
if os.name == "nt":  # Windows
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "Library", "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "Library", "share", "proj"))
else:  # Linux / macOS
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "share", "proj"))

import bw2data as bd
import bw2io as bi
import bw2regional as bwr
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterstats

from src.config import (
    PROJECT_NAME, DB_BIOSPHERE, DB_ECOINVENT, DB_FOREGROUND, BACKUP_PATH,
    COTTON_RASTER_PATH, LAND_USE_CFS_PATH, WATER_CFS_PATH, WATER_BASINS_PATH,
    N_CFS_PATH, P_CFS_PATH, CC_CFS_PATH,
    TZ_DISTRICTS_PATH, WWF_ECOREGIONS_PATH, GINNERIES_PATH, TEXTILE_PLANTS_PATH,
    GC_TZ_DISTRICTS, GC_WWF_ECOREGION, GC_COTTON_RASTER, XT_COTTON_PRODUCTION,
    METHOD_LAND_USE_REGIONAL, METHOD_LAND_USE_GENERIC,
    METHOD_WATER, METHOD_N_EUTRO, METHOD_P_EUTRO, METHOD_CLIMATE_CHANGE,
)
print("Imports OK")


C:\Users\elishaw\AppData\Local\miniconda3\envs\bw25-regional\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Imports OK

## 1. Restore Brightway project

In [2]:
# Restores ecoinvent 3.12 cutoff + biosphere-3.12 from the Activity Browser backup.
# overwrite_existing=True ensures a clean slate; takes ~15 min on first run.
bi.backup.restore_project_directory(
    fp=BACKUP_PATH,
    project_name=PROJECT_NAME,
    overwrite_existing=True,
    switch=True,
)
print("Active project:", bd.projects.current)
print("Available databases:", list(bd.databases))


Restoring project backup archive - this could take a few minutes...

Restored project: tz_cotton

Active project:

tz_cotton

Available databases:

['biosphere-3.12', 'ecoinvent-3.12-cutoff']

In [3]:
# Verify the two background databases are present and populated
biosphere = bd.Database(DB_BIOSPHERE)
ecoinvent = bd.Database(DB_ECOINVENT)
print(f"Biosphere flows:      {len(biosphere)}")
print(f"Ecoinvent activities: {len(ecoinvent)}")


Biosphere flows:      9850

Ecoinvent activities: 26533

## 2. Base geocollections (world, ecoinvent, RoW)

In [4]:
# Download base geocollections from bw2regional (only on first run).
# These provide the world-level spatial context needed by OneSpatialScaleLCA.
if "world" not in bwr.geocollections:
    bwr.create_world_collections()
    bwr.create_ecoinvent_collections()
    bwr.create_restofworlds_collections()
    print("Base geocollections created.")
else:
    print("Base geocollections already present, skipping download.")


Base geocollections already present, skipping download.

## 3. Load spatial datasets

In [5]:
tz_districts_gdf   = gpd.read_file(TZ_DISTRICTS_PATH)
wwf_ecoregions_gdf = gpd.read_file(WWF_ECOREGIONS_PATH)
water_basins_gdf   = gpd.read_file(WATER_BASINS_PATH)
ginneries_gdf      = gpd.read_file(GINNERIES_PATH)
textile_plants_gdf = gpd.read_file(TEXTILE_PLANTS_PATH)

# ECO_ID must be integer for joining with the CF table
wwf_ecoregions_gdf["ECO_ID"] = wwf_ecoregions_gdf["ECO_ID"].astype(int)

print(f"TZ districts:       {len(tz_districts_gdf)} features, CRS={tz_districts_gdf.crs}")
print(f"WWF ecoregions:     {len(wwf_ecoregions_gdf)} features")
print(f"PCR-GLOBWB basins:  {len(water_basins_gdf)} features")
print(f"Ginneries:          {len(ginneries_gdf)} features")
print(f"Textile plants:     {len(textile_plants_gdf)} features")


TZ districts:       170 features, CRS=EPSG:4326

WWF ecoregions:     14458 features

PCR-GLOBWB basins:  20318 features

Ginneries:          37 features

Textile plants:     7 features

## 4. Register Tanzania geocollections

In [6]:
bwr.geocollections[GC_TZ_DISTRICTS] = {
    "filepath": TZ_DISTRICTS_PATH,
    "field": "ADM2_PCODE",    # unique district code, e.g. "TZ001"
}
bwr.geocollections[GC_WWF_ECOREGION] = {
    "filepath": WWF_ECOREGIONS_PATH,
    "field": "ECO_ID",
}
bwr.geocollections[GC_COTTON_RASTER] = {
    "filepath": COTTON_RASTER_PATH,
}
print("Registered geocollections:", list(bwr.geocollections.keys()))


Registered geocollections:

['world', 'ecoinvent', 'tz_municipalities', 'wwf_ecoregions', 'cotton_production_raster', 'tz_districts']

## 5. Create extension table from SPAM 2020 cotton raster

In [7]:
# Zonal statistics: mean cotton production (kg/ha) per district polygon.
# Districts with zero or no cotton pixels are excluded.
stats = rasterstats.zonal_stats(
    tz_districts_gdf,
    COTTON_RASTER_PATH,
    stats=["mean"],
    all_touched=True,
)
field = "ADM2_PCODE"
xtable_data = [
    (row["mean"], (GC_TZ_DISTRICTS, gdf_row[field]))
    for row, gdf_row in zip(stats, tz_districts_gdf.to_dict("records"))
    if row.get("mean") is not None and row["mean"] > 0
]
xt = bwr.ExtensionTable(XT_COTTON_PRODUCTION)
xt.register(
    geocollections=[GC_TZ_DISTRICTS],
    data={"vector": GC_TZ_DISTRICTS, "raster": GC_COTTON_RASTER},
)
xt.write(xtable_data)
print(f"Extension table: {len(xtable_data)} districts with cotton production data")


C:\Users\elishaw\AppData\Local\miniconda3\envs\bw25-regional\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Extension table: 88 districts with cotton production data

## 6. Spatial joins (ginneries and textile plants to districts)

In [8]:
ginneries_proj = ginneries_gdf.to_crs(tz_districts_gdf.crs)
ginneries_by_district = gpd.sjoin(
    ginneries_proj,
    tz_districts_gdf[["ADM2_PCODE", "ADM2_EN", "geometry"]],
    how="left", predicate="within",
)
print(f"Ginneries joined: {len(ginneries_by_district)} rows")

textile_proj = textile_plants_gdf.to_crs(tz_districts_gdf.crs)
textile_by_district = gpd.sjoin(
    textile_proj,
    tz_districts_gdf[["ADM2_PCODE", "ADM2_EN", "geometry"]],
    how="left", predicate="within",
)
print(f"Textile plants joined: {len(textile_by_district)} rows")


Ginneries joined: 37 rows

Textile plants joined: 7 rows

## 7. Compute spatial weights

These are used by `build_foreground_db()` to distribute national demand across district activities.

In [9]:
# Production weights from SPAM raster
production_by_district = {
    district_id: value
    for value, (_, district_id) in xtable_data
}
total_production = sum(production_by_district.values())
production_weights = {d: v / total_production for d, v in production_by_district.items()}
print(f"Production weights: {len(production_weights)} districts (sum={sum(production_weights.values()):.4f})")

# Ginning weights from ginnery shapefile Production column
_SKIP_COLS = {"index_right", "SymbolID", "AltMode", "Base", "HasLabel", "LabelID"}
if "Production" in ginneries_by_district.columns:
    ginnery_prod_col = "Production"
else:
    _num = ginneries_by_district.select_dtypes(include=[float, int]).columns.tolist()
    _candidates = [c for c in _num if c not in _SKIP_COLS]
    ginnery_prod_col = _candidates[0] if _candidates else None

if ginnery_prod_col and ginnery_prod_col in ginneries_by_district.columns:
    agg = ginneries_by_district.groupby("ADM2_PCODE")[ginnery_prod_col].sum()
    total_gin = agg.sum()
    ginning_weights = {d: v / total_gin for d, v in agg.items() if v > 0} if total_gin > 0 else production_weights.copy()
else:
    gin_districts = ginneries_by_district["ADM2_PCODE"].dropna().unique()
    ginning_weights = {d: 1 / len(gin_districts) for d in gin_districts} if len(gin_districts) else production_weights.copy()
print(f"Ginning weights:    {len(ginning_weights)} districts (sum={sum(ginning_weights.values()):.4f})")

# Textile weights from plant count per district
if len(textile_by_district) > 0 and "ADM2_PCODE" in textile_by_district.columns:
    plant_counts = textile_by_district["ADM2_PCODE"].dropna().value_counts()
    total_plants = plant_counts.sum()
    textile_weights = {d: cnt / total_plants for d, cnt in plant_counts.items()} if total_plants > 0 else production_weights.copy()
else:
    textile_weights = production_weights.copy()
print(f"Textile weights:    {len(textile_weights)} districts (sum={sum(textile_weights.values()):.4f})")


Production weights: 88 districts (sum=1.0000)

Ginning weights:    22 districts (sum=1.0000)

Textile weights:    6 districts (sum=1.0000)

## 8. Calculate district × ecoregion intersection

In [10]:
bwr.calculate_intersection(
    GC_TZ_DISTRICTS,
    GC_WWF_ECOREGION,
    engine="geopandas",
    overwrite=False,
)
print(f"Intersection: {GC_TZ_DISTRICTS} x {GC_WWF_ECOREGION} done")


Intersection: tz_districts x wwf_ecoregions done

## 9. Land use CFs (LC-IMPACT)

In [11]:
# Load taxon-aggregated (Eukaryota) land use CFs from LC-IMPACT.
#
# CF_domain.csv aggregates the five species groups (amphibians, birds, mammals,
# plants, reptiles) into a single "Eukaryota" domain, giving exactly one row per
# (ecoregion, habitat). The previous CF.csv + str.contains("Cropland_Light")
# filter matched BOTH Cropland_Light (plants only) and Cropland_LightIntense
# (four vertebrate groups), averaging two different habitat classes across
# mismatched taxa.
#
# Two habitats are used:
#   Cropland_Light -> "Occupation, permanent crop, non-irrigated, intensive"
#                     (cotton farming)
#   Urban_Intense  -> "Occupation, industrial area"
#                     (ginning, oil milling, textile manufacturing sites)
HABITAT_AGRI  = "Cropland_Light"
HABITAT_INDUS = "Urban_Intense"

landuse_cf_raw = pd.read_csv(LAND_USE_CFS_PATH, encoding="latin1")

# Auto-detect column names
eco_id_candidates = ["eco_id", "ECO_ID", "ecoregion_id", "ECOREGION", "ID", "id"]
eco_id_col = next((c for c in eco_id_candidates if c in landuse_cf_raw.columns),
                  landuse_cf_raw.columns[0])
cf_value_col = "CF_occ_avg_reg"
if cf_value_col not in landuse_cf_raw.columns:
    numeric_cols = landuse_cf_raw.select_dtypes(include=[float, int]).columns.tolist()
    cf_value_col = next((c for c in numeric_cols if c.startswith("CF_")), None)
    if cf_value_col is None:
        raise ValueError("Cannot identify CF value column")
    print(f"WARNING: using fallback CF column: '{cf_value_col}'")

# Exact habitat match (NOT str.contains) so sibling classes such as
# Cropland_LightIntense are never picked up.
landuse_cf       = landuse_cf_raw[landuse_cf_raw["habitat"] == HABITAT_AGRI].copy()
landuse_cf_urban = landuse_cf_raw[landuse_cf_raw["habitat"] == HABITAT_INDUS].copy()

if landuse_cf.empty or landuse_cf_urban.empty:
    raise ValueError(
        f"Missing habitat rows: {HABITAT_AGRI}={len(landuse_cf)}, "
        f"{HABITAT_INDUS}={len(landuse_cf_urban)}"
    )

avg_cf       = landuse_cf[cf_value_col].mean()
avg_cf_urban = landuse_cf_urban[cf_value_col].mean()

print(f"Species groups in file: {sorted(landuse_cf_raw['species_group'].unique())}")
print(f"Eco-region ID column: '{eco_id_col}'   CF column: '{cf_value_col}'")
print(f"{HABITAT_AGRI:<16} rows={len(landuse_cf):<5} mean CF={avg_cf:.4e}")
print(f"{HABITAT_INDUS:<16} rows={len(landuse_cf_urban):<5} mean CF={avg_cf_urban:.4e}")
print(f"Urban/Cropland CF ratio: {avg_cf_urban / avg_cf:.3f}x")


Species groups in file: ['Eukaryota']

Eco-region ID column: 'eco_id'   CF column: 'CF_occ_avg_reg'

Cropland_Light   rows=825   mean CF=2.6856e-11

Urban_Intense    rows=825   mean CF=4.2875e-11

Urban/Cropland CF ratio: 1.596x

## 10. Area-weighted district land use CFs

In [12]:
# For each district: compute an area-weighted CF by intersecting with WWF
# ecoregions and weighting each ecoregion CF by the fraction of the district
# area it covers.
#
# Iterate over ALL districts (not just cotton-producing ones). Foreground
# activities other than cotton farming - ginning, textile mfg - sit in
# non-cotton-producing districts (e.g. Dar Es Salaam textile mills), and
# OneSpatialScaleLCA needs a CF for every district that hosts an activity.
# Districts with no overlapping ecoregion fall back to the country-mean CF.
#
# Geometries are first run through shapely.make_valid() to repair invalid
# topology (self-intersections, ring orientation issues, slivers).  Several
# districts along Lake Malawi / the Mozambique border have minor topology
# defects that trigger GEOSException on intersection.
from shapely.validation import make_valid

def _fix(g):
    return g if (g is not None and g.is_valid) else make_valid(g)

tz_districts_gdf["geometry"]   = tz_districts_gdf.geometry.apply(_fix)
wwf_ecoregions_gdf["geometry"] = wwf_ecoregions_gdf.geometry.apply(_fix)


def compute_district_cfs(cf_df, fallback_cf, label=""):
    """Area-weighted district CFs from ecoregion CFs for a single habitat."""
    out = {}
    for district_id in tz_districts_gdf["ADM2_PCODE"]:
        d_geom_s = tz_districts_gdf.loc[
            tz_districts_gdf["ADM2_PCODE"] == district_id, "geometry"]
        if d_geom_s.empty:
            continue
        d_geom = d_geom_s.values[0]

        overlapping = wwf_ecoregions_gdf[
            wwf_ecoregions_gdf.geometry.intersects(d_geom)].copy()
        if overlapping.empty:
            out[district_id] = fallback_cf
            continue

        overlapping["overlap_area"] = overlapping.geometry.apply(
            lambda g: d_geom.intersection(g).area)
        total_area = overlapping["overlap_area"].sum()

        weighted_cf = 0.0
        for _, eco_row in overlapping.iterrows():
            eco_id_val = str(int(eco_row["ECO_ID"]))
            cf_match = cf_df[cf_df[eco_id_col].astype(str) == eco_id_val]
            if not cf_match.empty:
                cf_val = cf_match[cf_value_col].values[0]
                weighted_cf += cf_val * (eco_row["overlap_area"] / total_area)

        out[district_id] = weighted_cf if weighted_cf > 0 else fallback_cf

    print(f"{label:<16} CFs for {len(out)} districts", end="")
    if out:
        print(f"   range {min(out.values()):.4e} - {max(out.values()):.4e}")
    else:
        print()
    return out


district_cfs       = compute_district_cfs(landuse_cf,       avg_cf,       HABITAT_AGRI)
district_cfs_urban = compute_district_cfs(landuse_cf_urban, avg_cf_urban, HABITAT_INDUS)


Cropland_Light   CFs for 170 districts

   range 8.2011e-13 - 3.5556e-11

Urban_Intense    CFs for 170 districts

   range 8.9735e-13 - 8.5378e-11

## 11. Register LCIA methods

In [13]:
# Resolve the land use biosphere flows (needed to register methods)
biosphere = bd.Database(DB_BIOSPHERE)

def _get_bio(name, cats):
    matches = [a for a in biosphere
               if a["name"] == name and tuple(a.get("categories", ())) == tuple(cats)]
    if not matches:
        matches = [a for a in biosphere if a["name"] == name]
    return matches[0]

landuse_agricultural = _get_bio("Occupation, permanent crop, non-irrigated, intensive",
                                ("natural resource", "land"))
# Emitted by ginning, oil milling and textile manufacturing (src/inventory.py).
# Previously had no CF in either land use method, so those stages scored
# exactly zero for land use.
landuse_industrial   = _get_bio("Occupation, industrial area",
                                ("natural resource", "land"))
water_from_river     = _get_bio("Water, river", ("natural resource", "in water"))
nitrite_emission     = _get_bio("Nitrite", ("water", "surface water"))
phosphorous_emission = _get_bio("Phosphorus", ("water", "surface water"))
print("Biosphere nodes resolved for method registration")
print(f"  agricultural: {landuse_agricultural['name']}")
print(f"  industrial:   {landuse_industrial['name']}")


Biosphere nodes resolved for method registration

  agricultural: Occupation, permanent crop, non-irrigated, intensive

  industrial:   Occupation, industrial area

In [14]:
# Site-generic land use method (single national-average CF per flow).
# Always (re)written so CF changes propagate; the method is small.
if METHOD_LAND_USE_GENERIC in bd.methods:
    bd.Method(METHOD_LAND_USE_GENERIC).deregister()

m = bd.Method(METHOD_LAND_USE_GENERIC)
m.register(unit="PDF*m2*year",
           description="Site-generic land use occupation impact (average CF, Eukaryota)",
           geocollections=[])
m.write([
    (landuse_agricultural.key, avg_cf),
    (landuse_industrial.key,   avg_cf_urban),
])
print(f"Site-generic method written:")
print(f"  agricultural CF = {avg_cf:.4e}")
print(f"  industrial   CF = {avg_cf_urban:.4e}")


Site-generic method written:

  agricultural CF = 2.6856e-11

  industrial   CF = 4.2875e-11

In [15]:
# Regionalized land use method keyed by tz_districts.
# Both land use flows are characterized: agricultural (cotton farming) and
# industrial (ginning, oil milling, textile manufacturing).
method_regional = bd.Method(METHOD_LAND_USE_REGIONAL)
method_regional.register(
    unit="PDF*m2*year",
    description=("Regionalized land use CFs by Tanzania districts (ADM2), "
                 "LC-IMPACT Eukaryota; agricultural + industrial occupation"),
    geocollections=[GC_TZ_DISTRICTS],
)
cfs_regional = [
    (landuse_agricultural.key, cf_value, (GC_TZ_DISTRICTS, district_id))
    for district_id, cf_value in district_cfs.items()
] + [
    (landuse_industrial.key, cf_value, (GC_TZ_DISTRICTS, district_id))
    for district_id, cf_value in district_cfs_urban.items()
]
method_regional.write(cfs_regional)
print(f"Regionalized land use method written: {len(cfs_regional)} CF entries "
      f"({len(district_cfs)} agricultural + {len(district_cfs_urban)} industrial)")


Regionalized land use method written: 340 CF entries (170 agricultural + 170 industrial)

In [16]:
# Water consumption method (PCR-GLOBWB basin CFs, area-weighted to districts)
#
# Iterate over ALL districts (not just cotton-producing ones), so that textile
# mills and ginneries sitting in non-cotton districts (e.g. Dar Es Salaam,
# Arusha) get characterised in the regionalized water LCA. Districts with no
# overlapping basin polygon fall back to the country-mean CF.
#
# Geometries are repaired via shapely.make_valid() — Tanzania districts along
# Lake Malawi / Mozambique border, and basin polygons spanning the same area,
# have minor topology defects that trigger GEOSException on intersection.
from shapely.validation import make_valid

def _fix(g):
    return g if (g is not None and g.is_valid) else make_valid(g)

water_basin_raw = pd.read_excel(WATER_CFS_PATH, sheet_name="Basin_CF")
water_basin_raw["CF_GLOB_M_m"] = pd.to_numeric(water_basin_raw["CF_GLOB_M_m"], errors="coerce")
basin_cf_lookup = dict(zip(water_basin_raw["id_basin_pcrglob"], water_basin_raw["CF_GLOB_M_m"]))

water_cf_raw = pd.read_excel(WATER_CFS_PATH, sheet_name="Country_CF")
water_cf_raw["CF_GLOB_M_m"] = pd.to_numeric(water_cf_raw["CF_GLOB_M_m"], errors="coerce")
water_cf_tz = float(water_cf_raw.loc[water_cf_raw["ISO3CD"] == "TZA", "CF_GLOB_M_m"].iloc[0])

water_basins_gdf["cf_water"] = water_basins_gdf["id_basin_pcrglob"].map(basin_cf_lookup)
basins_with_cf = water_basins_gdf[water_basins_gdf["cf_water"].notna()].copy()

tz_bounds = tz_districts_gdf.total_bounds
if tz_districts_gdf.crs.to_epsg() == 4326:
    buf = 1.0
    tz_bounds_4326 = tz_bounds
else:
    import pyproj
    transformer = pyproj.Transformer.from_crs(tz_districts_gdf.crs, "EPSG:4326", always_xy=True)
    x0, y0 = transformer.transform(tz_bounds[0], tz_bounds[1])
    x1, y1 = transformer.transform(tz_bounds[2], tz_bounds[3])
    tz_bounds_4326 = [x0, y0, x1, y1]
    buf = 1.0
basins_tz = basins_with_cf.cx[tz_bounds_4326[0]-buf : tz_bounds_4326[2]+buf,
                               tz_bounds_4326[1]-buf : tz_bounds_4326[3]+buf].copy()

tz_4326_w = tz_districts_gdf.to_crs("EPSG:4326") if tz_districts_gdf.crs.to_epsg() != 4326 else tz_districts_gdf

# Repair invalid geometries on the basin slice and the district frame
basins_tz["geometry"]  = basins_tz.geometry.apply(_fix)
tz_4326_w = tz_4326_w.copy()
tz_4326_w["geometry"]  = tz_4326_w.geometry.apply(_fix)

district_water_cfs = {}
for district_id in tz_districts_gdf["ADM2_PCODE"]:
    d_geom_s = tz_4326_w.loc[tz_4326_w["ADM2_PCODE"] == district_id, "geometry"]
    if d_geom_s.empty:
        continue
    d_geom = d_geom_s.values[0]
    overlapping = basins_tz[basins_tz.geometry.intersects(d_geom)].copy()
    if overlapping.empty:
        district_water_cfs[district_id] = water_cf_tz
        continue
    overlapping["overlap_area"] = overlapping.geometry.apply(lambda g: d_geom.intersection(g).area)
    total_area = overlapping["overlap_area"].sum()
    weighted_cf = (overlapping["cf_water"] * overlapping["overlap_area"]).sum() / total_area
    district_water_cfs[district_id] = weighted_cf if not np.isnan(weighted_cf) else water_cf_tz

method_water = bd.Method(METHOD_WATER)
try:
    method_water.register(
        unit="PDF*yr/m3",
        description="Freshwater consumption impact. PCR-GLOBWB 5-arcmin basin CFs (CF_GLOB_M_m) aggregated to TZ ADM2.",
        geocollections=[GC_TZ_DISTRICTS],
    )
except Exception:
    pass
cfs_water = [(water_from_river.key, cf, (GC_TZ_DISTRICTS, d)) for d, cf in district_water_cfs.items()]
method_water.write(cfs_water)
print(f"Water consumption method written: {len(cfs_water)} CF entries")


Water consumption method written: 170 CF entries

In [17]:
# N and P eutrophication methods (0.5-degree raster, zonal mean per district)
tz_4326 = tz_districts_gdf.to_crs("EPSG:4326") if tz_districts_gdf.crs.to_epsg() != 4326 else tz_districts_gdf

stats_n = rasterstats.zonal_stats(tz_4326, N_CFS_PATH, stats=["mean"], nodata=-9999.0, all_touched=True)
district_n_cfs = {
    row["ADM2_PCODE"]: stat["mean"]
    for stat, row in zip(stats_n, tz_4326.to_dict("records"))
    if stat["mean"] is not None and stat["mean"] > 0
}
method_n = bd.Method(METHOD_N_EUTRO)
try:
    method_n.register(unit="PDF*yr/kg_N",
                      description="Marginal direct N eutrophication CF (0.5-deg raster, zonal mean). Applied to Nitrite.",
                      geocollections=[GC_TZ_DISTRICTS])
except Exception:
    pass
cfs_n = [(nitrite_emission.key, cf, (GC_TZ_DISTRICTS, d)) for d, cf in district_n_cfs.items()]
method_n.write(cfs_n)
print(f"N eutrophication method: {len(cfs_n)} CF entries")

stats_p = rasterstats.zonal_stats(tz_4326, P_CFS_PATH, stats=["mean"], nodata=-9999.0, all_touched=True)
district_p_cfs = {
    row["ADM2_PCODE"]: stat["mean"]
    for stat, row in zip(stats_p, tz_4326.to_dict("records"))
    if stat["mean"] is not None and stat["mean"] > 0
}
method_p = bd.Method(METHOD_P_EUTRO)
try:
    method_p.register(unit="PDF*yr/kg_P",
                      description="Marginal direct P eutrophication CF (0.5-deg raster, zonal mean). Applied to Phosphorus.",
                      geocollections=[GC_TZ_DISTRICTS])
except Exception:
    pass
cfs_p = [(phosphorous_emission.key, cf, (GC_TZ_DISTRICTS, d)) for d, cf in district_p_cfs.items()]
method_p.write(cfs_p)
print(f"P eutrophication method: {len(cfs_p)} CF entries")


N eutrophication method: 154 CF entries

P eutrophication method: 124 CF entries

In [18]:
# Climate change method (NaturalEarth sheet, RCP2.6, global CFs)
COMPARTMENT_MAP = {
    "Emissions to air, unspecified":                  ("air",),
    "Emissions to urban air close to ground":         ("air", "urban air close to ground"),
    "Emissions to non-urban air or from high stacks": ("air", "non-urban air or from high stacks"),
    "Emissions to non-urban air high stack":          ("air", "non-urban air or from high stacks"),
    "Renewable material resources from air":          ("natural resource", "in air"),
}
# Replicate each LC-IMPACT row to additional biosphere3 air sub-compartments
# that share the same global CF (no spatial differentiation). Covers the
# rural-field "low population density, long-term" sub-compartment used by
# some ecoinvent activities. Field N₂O / urea-CO₂ in inventory.py target
# ("air", "non-urban air or from high stacks"), which is already in the
# base map above.
EXTRA_AIR_COMPARTMENTS = [
    ("air", "low population density, long-term"),
]
NAME_MAP = {
    "carbon dioxide":                     ["Carbon dioxide, fossil", "Carbon dioxide, non-fossil"],
    "carbon dioxide (fossil)":            ["Carbon dioxide, fossil"],
    "carbon dioxide (biogenic)":          ["Carbon dioxide, non-fossil"],
    "carbon dioxide (land use change)":   ["Carbon dioxide, from soil or biomass stock"],
    "methane":                            ["Methane, fossil", "Methane, non-fossil"],
    "methane (fossil)":                   ["Methane, fossil"],
    "methane (biogenic)":                 ["Methane, non-fossil"],
    "nitrous oxide":                      ["Dinitrogen monoxide"],
}

cc_raw = pd.read_excel(CC_CFS_PATH, sheet_name="NaturalEarth")
cc_rcp26 = cc_raw[
    (cc_raw["Scenario"] == "rcp26") &
    (cc_raw["Matching_Flow_Status"].isin(["Clear matching", "Matching assumed"]))
].copy()

bio_by_name_cats = {}
bio_lower_to_names = {}
for f in biosphere:
    bio_by_name_cats[(f.get("name", ""), tuple(f.get("categories", ())))] = f
    bio_lower_to_names.setdefault(f.get("name", "").lower(), set()).add(f.get("name", ""))

cfs_cc = []
for _, row in cc_rcp26.iterrows():
    base_cats = COMPARTMENT_MAP.get(str(row["FLOW_class2"]))
    if base_cats is None:
        continue
    name_lower = str(row["FLOW_name"]).strip().lower()
    cf_val = float(row["CF"])
    eco_names = NAME_MAP.get(name_lower) or list(bio_lower_to_names.get(name_lower, []))
    target_cats_list = [base_cats]
    if base_cats[0] == "air":
        target_cats_list = list(dict.fromkeys([base_cats] + EXTRA_AIR_COMPARTMENTS))
    for tcats in target_cats_list:
        for eco_name in eco_names:
            flow = bio_by_name_cats.get((eco_name, tcats))
            if flow:
                cfs_cc.append((flow.key, cf_val))
cfs_cc = list({entry[0]: entry for entry in cfs_cc}.values())

method_cc = bd.Method(METHOD_CLIMATE_CHANGE)
try:
    method_cc.register(
        unit="PDF*yr/kg_emission",
        description="Global terrestrial biodiversity loss from climate change. NaturalEarth, RCP2.6.",
    )
except Exception:
    pass
method_cc.write(cfs_cc)
print(f"Climate change method written: {len(cfs_cc)} CF entries")

Climate change method written: 36 CF entries

In [19]:
# Freshwater ecotoxicity method (USEtox 2.1 / LC-IMPACT v2, W6 continental box)
#
# Maps biosphere flows to CFs by CAS number using the W6 (North, West, East &
# Central Africa) box. Two conventions apply, both documented in src/config.py:
#   * metals are read from the 100-year sheet, organics from the long-term one
#   * ecoinvent ", long-term" subcompartments are excluded entirely ("no-LT")
# Coverage includes the foreground pesticides (profenofos, lambda-cyhalothrin)
# plus every background ecoinvent flow whose CAS appears in USEtox.
from src.config import (
    ECOTOX_CFS_PATH, METHOD_ECOTOX_FW, METHOD_ECOTOX_FW_SUPERSEDED,
    ECOTOX_SHEET_DEFAULT, ECOTOX_SHEET_METALS, ECOTOX_LT_MARKER,
    USETOX_CONTINENT_LABEL, ECOTOX_M3DAY_TO_PDFYR,
)

# Emission pathways we characterize, by column position. Columns 3-10 hold the
# eight freshwater-endpoint pathways and sit at identical positions in both
# sheets, so one set of indices serves both.
_PATHWAY_COL = {
    "airU":     5,   # Em.airU       -> freshwater
    "airC":     6,   # Em.airC       -> freshwater
    "water":    7,   # Em.fr.waterC  -> freshwater
    "seawater": 8,   # Em.sea waterC -> freshwater
    "natsoil":  9,   # Em.nat.soilC  -> freshwater
    "agrsoil": 10,   # Em.agr.soilC  -> freshwater
}


def _load_w6_block(sheet_name):
    """Return {CAS: {pathway: CF}} for the W6 section of one workbook sheet.

    Each sheet carries a 3-row hierarchical header followed by 25 continental
    blocks, each introduced by a section-header row. Reading with
    header=[0, 1, 2, 3] folds the "Global average" section header into the
    column index; every later section header then appears as a data row with
    the continent label in column 0 and NaN elsewhere, which is easy to detect.
    """
    df = pd.read_excel(ECOTOX_CFS_PATH, sheet_name=sheet_name, header=[0, 1, 2, 3])
    df.columns = ["::".join(str(x) for x in col) for col in df.columns]
    cols    = list(df.columns)
    id_col  = cols[0]
    cas_col = next(c for c in cols if "cas" in c.lower())

    def _to_float(x):
        try:
            v = float(x)
            return v if v == v else None
        except Exception:
            return None

    block, in_target = {}, False
    for _, row in df.iterrows():
        id_raw = row[id_col]
        if isinstance(id_raw, str) and id_raw.strip() not in ("", "nan"):
            if USETOX_CONTINENT_LABEL in id_raw:
                in_target = True
            elif in_target:
                break                      # walked past the W6 block; done
            else:
                in_target = False
            continue
        if not in_target:
            continue
        cas_raw = row[cas_col]
        if pd.isna(cas_raw):
            continue
        cas = str(cas_raw).strip()
        if cas in ("", "nan"):
            continue
        block[cas] = {p: _to_float(row[cols[i]]) for p, i in _PATHWAY_COL.items()}
    return block


print(f"Target continent: {USETOX_CONTINENT_LABEL!r}")
print(f"PDF*m3*day -> PDF*yr conversion factor: {ECOTOX_M3DAY_TO_PDFYR:.4e}")

cf_all    = _load_w6_block(ECOTOX_SHEET_DEFAULT)   # organics + metals, t = inf
cf_metals = _load_w6_block(ECOTOX_SHEET_METALS)    # 27 metals only, t = 100 yr

# Metals take the 100-year values; everything else keeps the long-term sheet.
cf_by_cas = dict(cf_all)
cf_by_cas.update(cf_metals)
print(f"USEtox W6 CFs loaded for {len(cf_by_cas)} CAS-identified substances "
      f"({len(cf_metals)} metals overridden from the 100-year sheet)")

# Sanity-check the two foreground pesticides (organics -> long-term sheet)
for cas, name in [("41198-08-7", "Profenofos"), ("91465-08-6", "Lambda-cyhalothrin")]:
    d = cf_by_cas.get(cas)
    if d:
        print(f"  {name}  agr-soil={d['agrsoil']:.2f}, surface-water={d['water']:.2f}, "
              f"airC={d['airC']:.2f}  [PDF*m3*day/kg, raw]")
    else:
        print(f"  WARNING: {name} (CAS {cas}) not found in W6 CFs")

# Sanity-check one metal to confirm the 100-year override took effect
_al_new, _al_old = cf_by_cas.get("22537-23-1"), cf_all.get("22537-23-1")
if _al_new and _al_old:
    print(f"  Al(III)   agr-soil={_al_new['agrsoil']:.4g}  "
          f"(100-yr sheet; long-term value was {_al_old['agrsoil']:.4g})")

# Explicit biosphere-compartment -> emission-pathway map. Every compartment in
# biosphere3 that carries a CAS-matched flow is listed, so there is no partial
# fallback: an unlisted compartment is reported as uncharacterized rather than
# silently scored as a freshwater discharge. That fallback previously sent both
# ('water', 'ground-, long-term') and ('water', 'ocean') to the freshwater
# pathway; the first is now excluded outright, the second uses Em.sea waterC.
_COMPARTMENT_PATHWAY = {
    ("air",):                                          "airC",
    ("air", "non-urban air or from high stacks"):      "airC",
    ("air", "lower stratosphere + upper troposphere"): "airC",
    ("air", "urban air close to ground"):              "airU",
    ("water",):                                        "water",
    ("water", "surface water"):                        "water",
    ("water", "ground-"):                              "water",
    ("water", "ocean"):                                "seawater",
    ("soil",):                                         "natsoil",
    ("soil", "agricultural"):                          "agrsoil",
    ("soil", "forestry"):                              "natsoil",
    ("soil", "industrial"):                            "natsoil",
}

# Resource extractions are not emissions and have no emission pathway.
_NON_EMISSION_COMPARTMENTS = {"natural resource"}

biosphere  = bd.Database(DB_BIOSPHERE)
cfs_ecotox = []
n_matched = n_no_cas = n_cas_no_cf = n_dropped_lt = n_unmapped = 0
unmapped_comps = set()

for flow in biosphere:
    cas = str(flow.get("CAS number", "") or "").strip()
    if not cas or cas == "nan":
        n_no_cas += 1
        continue
    cf_dict = cf_by_cas.get(cas) or cf_by_cas.get(cas.lstrip("0"))
    if cf_dict is None:
        n_cas_no_cf += 1
        continue

    comp = tuple(flow.get("categories") or ())
    if comp and comp[0] in _NON_EMISSION_COMPARTMENTS:
        continue

    # no-LT: drop emissions released >100 yr after the activity (landfill and
    # slag-heap leachate). See src/config.py for the rationale and magnitude.
    if any(ECOTOX_LT_MARKER in str(x) for x in comp):
        n_dropped_lt += 1
        continue

    pathway = _COMPARTMENT_PATHWAY.get(comp)
    if pathway is None:
        n_unmapped += 1
        unmapped_comps.add(comp)
        continue

    cf_val = cf_dict.get(pathway)
    if cf_val is None or cf_val <= 0:
        continue
    # Raw USEtox CF (PDF*m3*day/kg) -> PDF*yr/kg by dividing through the W6
    # freshwater volume x 365.25 days, so LCIA scores come out in PDF*yr.
    cfs_ecotox.append((flow.key, cf_val * ECOTOX_M3DAY_TO_PDFYR))
    n_matched += 1

cfs_ecotox = list({k: (k, v) for k, v in cfs_ecotox}.values())
print(f"\nBiosphere flows characterized: {len(cfs_ecotox)}")
print(f"  matched: {n_matched} | no-CAS: {n_no_cas} | CAS-no-CF: {n_cas_no_cf}")
print(f"  dropped as long-term emission: {n_dropped_lt} | unmapped compartment: {n_unmapped}")
if unmapped_comps:
    print(f"  unmapped compartments seen: {sorted(unmapped_comps)}")

# Deregister prior versions of this method, including superseded names
for _old in list(METHOD_ECOTOX_FW_SUPERSEDED) + [METHOD_ECOTOX_FW]:
    if _old in bd.methods:
        bd.Method(_old).deregister()
        print(f"Deregistered existing method {_old}")

method_ecotox = bd.Method(METHOD_ECOTOX_FW)
method_ecotox.register(
    unit="PDF*yr",
    description=(
        f"USEtox 2.1 / LC-IMPACT v2 freshwater ecotoxicity. "
        f"Continental zone: {USETOX_CONTINENT_LABEL}. "
        f"Metal CFs from the 100-year sheet, organics from the long-term "
        f"(infinite-horizon) sheet. ecoinvent long-term emission "
        f"subcompartments are excluded (no-LT). CFs converted from raw "
        f"PDF*m3*day/kg to PDF*yr/kg by multiplication with "
        f"1/(V_FW_W6 x 365.25) = {ECOTOX_M3DAY_TO_PDFYR:.4e}. "
        "Score is therefore directly in PDF*yr."
    ),
)
method_ecotox.write(cfs_ecotox)
print(f"Freshwater ecotoxicity method written: {len(cfs_ecotox)} CF entries (unit: PDF*yr)")


Target continent: 'W6 (North, West, East & Central Africa)'

PDF*m3*day -> PDF*yr conversion factor: 2.7379e-13

USEtox W6 CFs loaded for 3104 CAS-identified substances (27 metals overridden from the 100-year sheet)

  Profenofos  agr-soil=398.88, surface-water=436752.33, airC=4447.80  [PDF*m3*day/kg, raw]

  Lambda-cyhalothrin  agr-soil=485.35, surface-water=9813764.31, airC=30389.96  [PDF*m3*day/kg, raw]

  Al(III)   agr-soil=1.446e+04  (100-yr sheet; long-term value was 9.046e+05)


Biosphere flows characterized: 5319

  matched: 5319 | no-CAS: 667 | CAS-no-CF: 3471

  dropped as long-term emission: 241 | unmapped compartment: 0

Freshwater ecotoxicity method written: 5319 CF entries (unit: PDF*yr)

## 12. Set geocollections on background databases

In [20]:
for db_name, gcs in [
    (DB_BIOSPHERE, ["world"]),
    (DB_ECOINVENT, ["world", "ecoinvent", "RoW"]),
]:
    _current = bd.databases[db_name].get("geocollections")
    if not _current:
        bd.databases[db_name]["geocollections"] = gcs
        bd.databases.flush()
        print(f"Set geocollections on {db_name}: {gcs}")
    else:
        print(f"{db_name} geocollections already set: {_current}")


Set geocollections on biosphere-3.12: ['world']

ecoinvent-3.12-cutoff geocollections already set: ['world']

## 13. Diagnostic check

In [21]:
print("Registered LC-IMPACT methods:", [m for m in bd.methods if "LC-IMPACT" in str(m)])
print("Extension tables:", list(bwr.extension_tables))
print("Setup complete. Proceed to 01_scenarios.ipynb.")


Registered LC-IMPACT methods:

[('LC-IMPACT', 'Land Use', 'Occupation', 'Average', 'Site-generic'), ('LC-IMPACT', 'Land Use', 'Occupation', 'Regionalized Districts TZ'), ('LC-IMPACT', 'Water Consumption', 'Freshwater Ecosystem Quality', 'Regionalized Districts TZ'), ('LC-IMPACT', 'Climate Change', 'Terrestrial Biodiversity', 'Global rcp26'), ('USEtox 2.1 / LC-IMPACT', 'Ecotoxicity', 'Freshwater', 'W6 N-W-E-C Africa, metals 100yr, no-LT')]

Extension tables:

['cotton_production_output']

Setup complete. Proceed to 01_scenarios.ipynb.